# 3. Od kodów D do wzorców F

[![Otwórz w Colabie](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caqdastm/ai_qda-workshop-1u/blob/main/04_vibe_coding/03_od_kodow_D_do_wzorcow_F.ipynb)

Cel: porównać profile D i zaproponować F tylko wtedy, gdy istnieje wspólny mechanizm lub proces.

To jest ćwiczenie z **projektowania pipeline'u kodowania AI_QDA**.
Nie odtwarza autorskiego generatora pełnego wyniku. Kod techniczny jest
zwinięty; widoczne pozostają materiał, karta procedury, odpowiedzi modelu
i decyzja badacza.

W promptach **CZĘŚĆ BADAWCZA** pochodzi z karty uczestnika, a
**DODATEK TECHNICZNY** tylko dopasowuje jedną funkcję do notebooka.


## Rytm pracy

`profile D → kryterium zysku analitycznego → dwa sposoby grupowania →
powrót do cytatów → split/merge/review → jawne mapowanie D–F`


## Dwa porządki pracy — nie mieszamy ich

**1. Tworzenie kodu:** krótką funkcję projektujesz w czacie AI
zintegrowanym z Colabem. Wysyłasz tam instrukcję o procedurze i
kontrakcie funkcji, a otrzymany kod wklejasz do wskazanej komórki.

**2. Analiza materiału:** dopiero działający notebook wysyła prompty i
ograniczony pakiet fragmentów przez API. W formularzu możesz wybrać
`gemini` albo `openai`; dalsze komórki pozostają takie same.

Dla wybranego providera dodaj w Colab Secrets tylko odpowiedni klucz:
`GEMINI_API_KEY` albo `OPENAI_API_KEY`. Przy OpenAI pole
`OPENAI_STORE` pozostaje jawną decyzją uczestnika; adapter zapisze jego
wartość i identyfikator odpowiedzi w lokalnym logu przebiegu.

Tryb `mock` sprawdza przepływ bez wysyłania danych. Czat Colaba służy do
vibe codingu, a `analysis_api` wyłącznie do porównań analitycznych.


In [ ]:
# @title Infrastruktura warsztatu — uruchom bez edycji { display-mode: "form" }
%pip install -q pandas "google-genai>=2.0.0" "openai>=2.0.0"

from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
from IPython.display import display

REPOSITORY_SLUG = "caqdastm/ai_qda-workshop-1u" # @param {type:"string"}
repo_folder = REPOSITORY_SLUG.replace("/", "__")
REPO_ROOT = Path("/content") / repo_folder
if not REPO_ROOT.exists():
    subprocess.run(
        ["git", "clone", "-q", f"https://github.com/{REPOSITORY_SLUG}.git", str(REPO_ROOT)],
        check=True,
    )
%cd $REPO_ROOT
support_dir = REPO_ROOT / "04_vibe_coding"
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))

from workshop_support import (
    AnalysisAPI,
    load_dataframe,
    load_workshop_packet,
    procedure_prompt,
    save_dataframe,
    save_json,
)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    WORKSPACE = Path("/content/drive/MyDrive/AI_QDA_Workshop")
except Exception:
    WORKSPACE = REPO_ROOT / "06_outputs" / "uczestnicy" / "AI_QDA_Workshop"
WORKSPACE.mkdir(parents=True, exist_ok=True)
print("Katalog przekazania między blokami:", WORKSPACE)


In [ ]:
# @title API analityczne — zmiana providera nie zmienia dalszych komórek { display-mode: "form" }
PROVIDER = "mock" # @param ["mock", "gemini", "openai"]
GEMINI_MODEL = "gemini-3.6-flash" # @param {type:"string"}
OPENAI_MODEL = "gpt-5.4-mini" # @param {type:"string"}
OPENAI_STORE = True # @param {type:"boolean"}
AUTHORIZE_API_CALLS = False # @param {type:"boolean"}
MAX_API_CALLS = 2 # @param {type:"integer"}

analysis_api = AnalysisAPI(
    provider=PROVIDER,
    gemini_model=GEMINI_MODEL,
    openai_model=OPENAI_MODEL,
    openai_store=OPENAI_STORE,
    authorize_api_calls=AUTHORIZE_API_CALLS,
    max_api_calls=MAX_API_CALLS,
)
print("Provider:", PROVIDER, "| model:", analysis_api.model, "| limit:", MAX_API_CALLS)


In [ ]:
# @title Wczytaj kandydackie D z bloku 2 { display-mode: "form" }
d_assignments = load_dataframe(
    WORKSPACE / "02_d_assignments.csv",
    ["code_id", "code_name", "text_unit_id", "evidence_quote", "review_status"],
)
d_profiles = d_assignments.loc[d_assignments["code_name"].astype(str).str.strip().ne("")].copy()
if d_profiles.empty:
    raise ValueError("Najpierw wróć do bloku 2 i zapisz co najmniej jeden kandydacki kod D.")
display(d_profiles[["code_id", "code_name", "evidence_quote", "memo"]])


In [ ]:
# @title Karta procedury kodowania zogniskowanego { display-mode: "form" }
cel = "Rozpoznać wzorce F, które wnoszą zysk analityczny ponad pojedyncze D." # @param {type:"string"}
kryterium_wspolnego_mechanizmu = "D opisują powiązane działania, warunki lub konsekwencje tworzące ten sam proces, a nie tylko podobne słowa." # @param {type:"string"}
wymagana_granica = "Każda F wskazuje, co do niej nie należy i czym różni się od sąsiedniego wzorca." # @param {type:"string"}
decyzja_badacza = "Czy wspólny mechanizm jest przekonujący; czy propozycję scalić, rozdzielić albo pozostawić w review." # @param {type:"string"}
PROCEDURE_CARD = {
    "goal": cel,
    "input": "Profile D zawierające nazwę, cytat i memo.",
    "observable_result": "Kandydackie F, ich rationale, granice, przypadki negatywne i jawne mapowanie D–F.",
    "automatic_check": "Każdy D ma jedną decyzję F albo needs_review; wskazane ID istnieją.",
    "researcher_decision": decyzja_badacza,
}


## Vibe coding w czacie Colaba: `find_unmapped_d`

1. Uruchom następną komórkę, aby wyświetlić instrukcję.
2. Otwórz panel czatu AI w Colabie i wklej całą instrukcję.
3. Poproś najpierw o krótkie powtórzenie kontraktu zwykłym językiem,
   a następnie o jedną funkcję — bez przebudowy notebooka.
4. Wklej otrzymaną funkcję do komórki **KOMÓRKA UCZESTNIKA**.

Na tym etapie nie korzystasz z klucza API i nie prosisz API
analitycznego o napisanie kodu.


In [ ]:
# @title Wyświetl instrukcję dla czatu Colaba { display-mode: "form" }
appendix = """Napisz funkcję find_unmapped_d(d_assignments, d_to_f_map).
Zwróć listę code_id D bez dokładnie jednej decyzji: istniejąca F albo
needs_review. Nie oceniaj podobieństwa, mechanizmu ani jakości F."""
FUNCTION_PROMPT = procedure_prompt(PROCEDURE_CARD, appendix)
print(FUNCTION_PROMPT)


In [ ]:
# KOMÓRKA UCZESTNIKA: wklej pełną funkcję otrzymaną od modelu.
find_unmapped_d = None


## Analiza korpusu przez API

Teraz kod pomocniczy jest już w notebooku. Dwa kolejne wywołania API
dostają ten sam materiał, ale inaczej sformułowane zadania analityczne.
Porównujesz wpływ promptu, a nie SDK providera. Zmiana `gemini` na
`openai` odbywa się wyłącznie w formularzu **API analityczne**.


In [ ]:
# @title Dwa wywołania analityczne: podobieństwo i wspólny mechanizm { display-mode: "form" }
material = d_profiles[["code_id", "code_name", "evidence_quote", "memo"]].to_csv(index=False)
prompt_a = f"""Połącz podobne kody D w szersze kategorie F. Podaj mapowanie.\n\n{material}"""
prompt_b = f"""Porównaj pełne profile D. Utwórz F tylko, gdy spełniają kryterium:
{kryterium_wspolnego_mechanizmu}
D może pozostać needs_review. Dla każdej F podaj: wspólny mechanizm,
zysk analityczny, granicę, przypadek negatywny i mapowanie ID.
Wszystkie propozycje mają status candidate.\n\n{material}"""
response_a = analysis_api.run_analysis(prompt_a, task_label="03_lexical_grouping")
response_b = analysis_api.run_analysis(prompt_b, task_label="03_mechanism_grouping")
display(pd.DataFrame([
    {"wariant": "A — podobieństwo", "odpowiedź": response_a},
    {"wariant": "B — mechanizm i granica", "odpowiedź": response_b},
]))


## Powrót do materiału i decyzja badacza

Odpowiedzi API są kandydackie. Wróć do cytatów i zapisz własną decyzję
w formularzu poniżej. Checklista może wykryć błąd struktury, ale nie
potwierdza trafności kodu, kategorii ani granicy interpretacji.


In [ ]:
# @title Zapisz dwie decyzje F po powrocie do cytatów { display-mode: "form" }
f1_name = "" # @param {type:"string"}
f1_d_ids = "" # @param {type:"string"}
f1_rationale = "" # @param {type:"string"}
f1_boundary = "" # @param {type:"string"}
f1_negative = "" # @param {type:"string"}
f2_name = "" # @param {type:"string"}
f2_d_ids = "" # @param {type:"string"}
f2_rationale = "" # @param {type:"string"}
f2_boundary = "" # @param {type:"string"}
f2_negative = "" # @param {type:"string"}
review_d_ids = "" # @param {type:"string"}

proposals = [
    ("F01", f1_name, f1_d_ids, f1_rationale, f1_boundary, f1_negative),
    ("F02", f2_name, f2_d_ids, f2_rationale, f2_boundary, f2_negative),
]
focused_rows, map_rows = [], []
for f_id, name, raw_ids, rationale, boundary, negative in proposals:
    ids = [value.strip() for value in raw_ids.split(",") if value.strip()]
    if name.strip():
        focused_rows.append({
            "focused_id": f_id, "focused_name": name.strip(),
            "analytic_rationale": rationale.strip(), "boundary": boundary.strip(),
            "negative_case": negative.strip(), "review_status": "candidate",
        })
        map_rows.extend({"code_id": d_id, "focused_id": f_id, "decision": "candidate"} for d_id in ids)
map_rows.extend({
    "code_id": d_id, "focused_id": "", "decision": "needs_review"
} for d_id in [value.strip() for value in review_d_ids.split(",") if value.strip()])
focused = pd.DataFrame(focused_rows, columns=[
    "focused_id", "focused_name", "analytic_rationale", "boundary", "negative_case", "review_status"
])
d_to_f = pd.DataFrame(map_rows, columns=["code_id", "focused_id", "decision"])
display(focused)
display(d_to_f)


In [ ]:
# @title Rozwiązanie awaryjne i kontrola kompletności { display-mode: "form" }
def prepared_find_unmapped_d(d_assignments, d_to_f_map):
    expected = set(d_assignments.loc[d_assignments["code_name"].astype(str).str.strip().ne(""), "code_id"])
    focused_ids = set(focused["focused_id"])
    decision_count = {}
    for d_id in expected:
        decisions = d_to_f_map.loc[d_to_f_map["code_id"].eq(d_id)]
        valid = decisions.apply(
            lambda row: (
                row["decision"] == "needs_review" and not str(row["focused_id"]).strip()
            ) or (
                row["decision"] == "candidate" and row["focused_id"] in focused_ids
            ),
            axis=1,
        ) if not decisions.empty else pd.Series(dtype=bool)
        decision_count[d_id] = int(valid.sum())
    return sorted(d_id for d_id, count in decision_count.items() if count != 1)
if not callable(globals().get("find_unmapped_d")):
    find_unmapped_d = prepared_find_unmapped_d
unmapped = find_unmapped_d(d_profiles, d_to_f)
display(pd.DataFrame({"D wymagające decyzji F albo review": unmapped}))
print("Brak na liście nie dowodzi trafności wspólnego mechanizmu F.")


In [ ]:
# @title Zapisz artefakty i przekazanie do bloku 4 { display-mode: "form" }
save_dataframe(WORKSPACE / "03_focused_categories.csv", focused)
save_dataframe(WORKSPACE / "03_d_to_f.csv", d_to_f)
save_json(WORKSPACE / "03_procedure_card.json", PROCEDURE_CARD)
save_json(WORKSPACE / "03_review_summary.json", {"unmapped_or_multiple": unmapped})
analysis_api.export_runs(WORKSPACE / "03_prompt_runs.jsonl")
print("Zapisano blok 3 w", WORKSPACE)


## Handoff

Blok 4 tworzy operacyjną kartę S z kilku zgodnych F. Nie traktuje F
bez decyzji jako błędu, który trzeba automatycznie ukryć.
